# LLM Multi-Seed Evaluation (Batch 1/2) — Qwen2.5-7B-Instruct × 3 Seeds [42, 123, 2024]

**Authors:** Swagotam Malakar, Anamika Das, Dr. Ohidujjaman
**Dataset:** Swarabyanjan Clean Balanced Dataset (766 samples: 383 yellow + 383 non-yellow)
**Environment:** Kaggle T4 ×2, Internet enabled, commit mode
**Estimated runtime:** ~7.5 hours on Kaggle T4 GPU (3 runs × ~2.5 h each)

---

## Purpose

This notebook is **Batch 1 of 2** of the LLM Multi-Seed Evaluation for **Qwen2.5-7B-Instruct** (the best-performing LLM in our benchmark). It runs the model with **3 different random seeds** `[42, 123, 2024]` to estimate run-to-run variance.

**This notebook is Batch 1 of 2.** Run NB9b for the remaining 2 seeds `[7, 99]`. Then run NB9c to aggregate all 5 seeds. Split into batches because the full 5-seed run (~13h) exceeds Kaggle's 12-hour commit limit.

We pick Qwen2.5-7B-Instruct because:

- It was the best-performing LLM (F1 = 0.075, the only one above 0.05)
- 7B is large enough to be representative of the LLM failure mode
- It is small enough to run 3× in ~7.5 hours on a Kaggle T4 (each run ~2.5 h), well under the 12-hour commit limit

## Reproducibility

All seeds are pinned; rerunning on Kaggle with the same inputs produces identical results.

## Hyperparameters (identical to NB5)

QLoRA 4-bit NF4 quantisation + LoRA (r=16, α=32), 3 epochs, batch size 2 with gradient accumulation 4, learning rate 2e-4 with cosine schedule and 3% warmup, max sequence length 320 (7B-specific override), 80/20 stratified split (612 train / 154 test). The only change vs NB5 is the seed: NB5 uses `seed=42` only; NB9a runs `[42, 123, 2024]` (NB9b runs `[7, 99]`).

## Required Kaggle Inputs

### 1. Gold Dataset (mandatory)
- **Search:** `swarabyanjan` or `v18-human-gold-final`
- **File:** `Swarabyanjan_BEST_BALANCED_1to1.csv`

### 2. Kaggle Model (preferred)
- **Qwen2.5-7B-Instruct:** `/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/*`
- If not available on Kaggle, the notebook falls back to HuggingFace Hub download (`hf_transfer` accelerates this 2–3×).

### 3. HuggingFace Token (optional, only if HF fallback used)
- **Secret name:** `HF_TOKEN`

## Execution Architecture

| Feature | Implementation |
|---------|----------------|
| **Parallelism** | Sequential across seeds (QLoRA 4-bit incompatible with DataParallel) |
| **GPU** | Single T4 (`CUDA_VISIBLE_DEVICES=0`); second T4 idle for stability |
| **Precision** | `float16` (T4 Turing architecture does not support `bfloat16`) |
| **Logging** | All prints use `flush=True`; intermediate results saved after each seed (crash-safe) |
| **Disk management** | HF cache and per-seed LoRA checkpoints deleted after each run |
| **Resumability** | `multi_seed_qwen7b_batch1_results.csv` is rewritten after each seed; partial CSVs are valid |
| **Error isolation** | Each seed is wrapped in `try/except`; a single failure does not terminate the experiment |


### 1. Environment Setup and Library Imports

The Kaggle T4 environment ships with PyTorch, transformers, and scikit-learn pre-installed. The `bitsandbytes`, `trl`, `peft`, `accelerate`, and `hf_transfer` packages are installed at the start of the run. `hf_transfer` provides 2-3x download acceleration when the HuggingFace fallback path is used.


In [1]:
%%capture _install
!pip install -q bitsandbytes trl peft accelerate datasets hf_transfer

# --- GPU isolation ---
# QLoRA 4-bit models (bitsandbytes) cannot use DataParallel — their parameters
# are pinned to a single device. Setting CUDA_VISIBLE_DEVICES=0 makes only
# one GPU visible to PyTorch, preventing DataParallel errors.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # 2-3x faster HuggingFace download
# Fix for CUDA OOM — expandable segments reduce fragmentation (per PyTorch error msg)
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import numpy as np
import pandas as pd
import gc, time, warnings, json, sys, re, glob, shutil
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, Tuple
import inspect

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, set_seed
)
from peft import LoraConfig, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer

warnings.filterwarnings('ignore')
set_seed(42)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

def _now():
    """Timestamp helper for logging."""
    return datetime.now().strftime('%H:%M:%S')

print(f"[{_now()}] PyTorch {torch.__version__}, CUDA={torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"[{_now()}] GPUs visible: {n_gpus}", flush=True)
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} (compute {props.major}.{props.minor})", flush=True)
else:
    print(f"[{_now()}] WARNING: No GPU detected.", flush=True)


### 2. Configuration — Seeds, Model Registry, and Hyperparameters

This is **Batch 1 of 2**: `SEEDS = [42, 123, 2024]` (NB9b runs the remaining 2 seeds `[7, 99]`). All hyperparameters (MAX_SEQ_LEN, NUM_EPOCHS, BATCH_SIZE, GRAD_ACCUM, LEARNING_RATE, LORA_R, LORA_ALPHA, TRAIN_FRAC) and the SYSTEM_PROMPT are identical to NB5.


In [2]:
# ============================================================
# CONFIGURATION — Multi-seed Qwen2.5-7B-Instruct (NB9a, Batch 1/2)
# ============================================================

SEEDS         = [42, 123, 2024]              # Batch 1 of 2 (NB9b: [7, 99])
TOTAL_SEEDS_PLANNED = 5                       # full multi-seed set across both batches
REMAINING_SEEDS = [7, 99]                     # seeds handled by NB9b
MAX_SEQ_LEN   = 512                           # global default; overridden to 320 for 7B
NUM_EPOCHS    = 3                             # same as NB5
BATCH_SIZE    = 2                             # global default; overridden to 1 for 7B
GRAD_ACCUM    = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO  = 0.03
LORA_R        = 16
LORA_ALPHA    = 32
TRAIN_FRAC    = 0.8                           # 612 train / 154 test (766 total)

GOLD_CSV_FILENAME = 'Swarabyanjan_Gold_Balanced_766.csv'  # Cleaned (Task 8): NFC + ZWJ stripped
GOLD_CSV_FILENAME_LEGACY = 'Swarabyanjan_BEST_BALANCED_1to1.csv'  # Legacy fallback
RESULTS_FILE  = '/kaggle/working/multi_seed_qwen7b_batch1_results.csv'
SUMMARY_FILE  = '/kaggle/working/multi_seed_qwen7b_batch1_summary.json'
OUTPUT_DIR    = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "You are a Bengali news analyst. Classify the given news article as "
    "yellow journalism (1) or not (0). Answer with only the digit 1 or 0."
)

# --- Single-model registry: only Qwen2.5-7B-Instruct for NB9a ---
# (Same fields as NB5's MODEL_CONFIGS entries, just a single dict.)
MODEL_CONFIG = {
    "name": "Qwen2.5-7B-Instruct",
    "hf_id": "Qwen/Qwen2.5-7B-Instruct",
    "kaggle_paths": ["/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/*"],
}

set_seed(SEEDS[0])  # initial seed before anything loads

def find_model_path(cfg):
    """Find local model path. Returns (path, source) tuple.
    source: 'kaggle' or 'huggingface'.
    """
    for pattern in cfg.get("kaggle_paths", []):
        matches = sorted(glob.glob(pattern))
        if matches:
            # Verify it has model files
            model_path = matches[0]
            has_safetensors = any(
                f.endswith('.safetensors') or f.endswith('.bin')
                for dp, _, fns in os.walk(model_path)
                for f in fns
            )
            if has_safetensors:
                return model_path, "kaggle"
    return cfg["hf_id"], "huggingface"

print(f"[{_now()}] NB9a: Multi-seed Qwen2.5-7B-Instruct (Batch 1/2)", flush=True)
print(f"[{_now()}] Seeds in this batch: {SEEDS}", flush=True)
print(f"[{_now()}] Total seeds planned (both batches): {TOTAL_SEEDS_PLANNED}", flush=True)
print(f"[{_now()}] Remaining seeds (handled by NB9b): {REMAINING_SEEDS}", flush=True)
print(f"[{_now()}] Hyperparameters: epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, "
      f"grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}, max_len={MAX_SEQ_LEN}, "
      f"lora_r={LORA_R}, lora_alpha={LORA_ALPHA}", flush=True)
print(f"[{_now()}] Results file: {RESULTS_FILE}", flush=True)
print(f"[{_now()}] Summary file: {SUMMARY_FILE}", flush=True)
print(f"[{_now()}] Checking model availability...", flush=True)
path, source = find_model_path(MODEL_CONFIG)
icon = "KAGGLE" if source == "kaggle" else "HF-HUB"
print(f"  [{icon}] {MODEL_CONFIG['name']:<28} -> {path}", flush=True)


[17:41:59] NB9a: Multi-seed Qwen2.5-7B-Instruct (Batch 1/2)
[17:41:59] Seeds in this batch: [42, 123, 2024]
[17:41:59] Total seeds planned (both batches): 5
[17:41:59] Remaining seeds (handled by NB9b): [7, 99]
[17:41:59] Hyperparameters: epochs=3, batch=2, grad_accum=4, lr=0.0002, max_len=512, lora_r=16, lora_alpha=32
[17:41:59] Results file: /kaggle/working/multi_seed_qwen7b_batch1_results.csv
[17:41:59] Summary file: /kaggle/working/multi_seed_qwen7b_batch1_summary.json
[17:41:59] Checking model availability...
  [KAGGLE] Qwen2.5-7B-Instruct          -> /kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1


### 3. GPU Detection and Disk Space Audit

The T4 GPU (Turing architecture, compute capability 7.5) does not support `bfloat16` operations natively; `float16` is used throughout. Disk space is audited before training begins, as 4-bit quantised models consume 2-6 GB each and the HuggingFace cache can grow rapidly across 3 sequential runs.


In [3]:
# GPU and disk audit
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader
    print(f"\ntorch.cuda.device_count() = {torch.cuda.device_count()}", flush=True)

# Disk space check (Kaggle /kaggle/working is the writable scratch volume)
disk_path = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
disk = os.statvfs(disk_path)
free_gb = (disk.f_bavail * disk.f_frsize) / 1e9
total_gb = (disk.f_blocks * disk.f_frsize) / 1e9
print(f"[{_now()}] Disk space: {free_gb:.1f} GB free / {total_gb:.1f} GB total", flush=True)
print(f"[{_now()}] Working directory: {disk_path}", flush=True)


0, Tesla T4, 15360 MiB, 14909 MiB
1, Tesla T4, 15360 MiB, 14912 MiB

torch.cuda.device_count() = 1
[17:42:00] Disk space: 20.9 GB free / 21.0 GB total
[17:42:00] Working directory: /kaggle/working


### 4. Dataset Loading with Path Auto-Discovery

The clean balanced dataset CSV is located by searching a prioritised list of candidate Kaggle mount paths, followed by a recursive glob search under `/kaggle/input/`. The headline and body preview columns are concatenated into a single text field, as this combination provides the full article context required for yellow journalism classification. The dataset is loaded once globally; per-seed train/test splits are performed inside `finetune_and_evaluate()`.


In [4]:
def find_gold_csv():
    """Find the gold standard CSV. Tries cleaned filename first, then legacy."""
    # Try cleaned filename first (Task 8 output)
    for fname in [GOLD_CSV_FILENAME, GOLD_CSV_FILENAME_LEGACY]:
        candidates = [
            f'/kaggle/input/datasets/smalakarishere/swarabyanjan/{fname}',
            f'/kaggle/input/swarabyanjan/{fname}',
            f'/kaggle/input/datasets/swagotammalakar/v18-human-gold-final/{fname}',
            f'/kaggle/input/v18-human-gold-final/{fname}',
            f'/kaggle/input/{fname}',
        ]
        for c in candidates:
            if os.path.isfile(c):
                print(f"[{{_now()}}] Found gold CSV: {{c}}", flush=True)
                return c
        # Recursive glob search
        matches = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        if matches:
            print(f"[{_now()}] Found gold CSV via glob: {matches[0]}", flush=True)
            return matches[0]
    # Also try wildcard for any Swarabyanjan gold CSV
    matches = glob.glob('/kaggle/input/**/Swarabyanjan_Gold_Balanced*.csv', recursive=True)
    if matches:
        print(f"[{_now()}] Found gold CSV via wildcard: {matches[0]}", flush=True)
        return matches[0]
    matches = glob.glob('/kaggle/input/**/Swarabyanjan_BEST_BALANCED*.csv', recursive=True)
    if matches:
        print(f"[{_now()}] Found legacy gold CSV via wildcard: {matches[0]}", flush=True)
        return matches[0]
    # Local fallback (for testing outside Kaggle)
    for local_fname in [GOLD_CSV_FILENAME, GOLD_CSV_FILENAME_LEGACY]:
        local = f'/home/z/my-project/analysis/github_repo/data/{local_fname}'
        if os.path.isfile(local):
            print(f"[{_now()}] Found gold CSV locally: {local}", flush=True)
            return local
        local = f'/home/z/my-project/download/{local_fname}'
        if os.path.isfile(local):
            print(f"[{_now()}] Found gold CSV in download: {local}", flush=True)
            return local
    print(f"[{_now()}] WARNING: Could not find gold CSV. Tried: {GOLD_CSV_FILENAME}, {GOLD_CSV_FILENAME_LEGACY}", flush=True)
    return f'/kaggle/input/{GOLD_CSV_FILENAME}'  # Return default path (will fail with helpful error)

DATA_PATH = find_gold_csv()
print(f"[{_now()}] Dataset: {DATA_PATH}", flush=True)
print(f"[{_now()}] Exists: {os.path.isfile(DATA_PATH)}", flush=True)

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['headline', 'body_text', 'best_label']).reset_index(drop=True)
df['best_label'] = df['best_label'].astype(int)
df['article'] = df['headline'].fillna('').astype(str) + '\n\n' + df['body_text'].fillna('').astype(str)
TEXT_COL = 'article'
LABEL_COL = 'best_label'

print(f"[{_now()}] Loaded: {df.shape}", flush=True)
print(f"Label distribution:\n{df[LABEL_COL].value_counts().sort_index().to_string()}", flush=True)


[{_now()}] Found gold CSV: {c}
[17:42:00] Dataset: /kaggle/input/datasets/smalakarishere/swarabyanjan/Swarabyanjan_Gold_Balanced_766.csv
[17:42:00] Exists: True
[17:42:00] Loaded: (766, 9)
Label distribution:
best_label
0    383
1    383


### 5. Define `finetune_and_evaluate(model_id, model_name, seed)`

Self-contained pipeline that runs one full fine-tune + evaluate cycle for a single seed. The function:

1. Calls `set_seed(seed)` first, so Python's `random`, `numpy`, and `torch` RNGs are all deterministic for this run.
2. Resolves the model path (Kaggle Input preferred, HuggingFace Hub fallback) using the global `MODEL_CONFIG`.
3. Applies the 7B-specific memory tuning (`MAX_SEQ_LEN=320`, `BATCH_SIZE=1`) — same as NB5's `run_single_model` does for 7B models.
4. Performs a stratified 80/20 train/test split with `random_state=seed` (so each seed gets a *different* split — this is the source of run-to-run variance).
5. Loads the tokenizer (with chat-template fallback for tokenizers that don't define one), the 4-bit NF4 model, sets up LoRA, trains with `TrainingArguments(seed=seed, data_seed=seed)`, evaluates with greedy decoding, plots a per-seed confusion matrix, and aggressively cleans up GPU memory and disk before returning.
6. Returns a metrics dict that includes a `"seed"` field set by the caller.

The `SFTTrainer` instantiation is version-safe (constructor signature is inspected and only supported parameters are passed), matching NB5 exactly.


In [5]:
# ============================================================
# Utility functions (copied from NB5) — metrics, parsing, cleanup
# ============================================================

def parse_prediction(text):
    """Parse LLM output to extract a 0/1 prediction.

    Strategy (in order):
    1. Strip whitespace and special tokens
    2. Check if first non-space char is "1" or "0"
    3. Check for Bengali yes/no words (হ্যাঁ/না)
    4. Check for English yes/no words
    5. Scan for Bengali digits ১/০
    6. Look for the LAST ASCII digit 0/1 in the text (LLMs often reason
       then conclude: "...therefore 1")
    7. Fall back to -1 (unparseable, defaults to 0 in the eval loop)
    """
    if not text or not text.strip():
        return -1

    text = text.strip()

    # Step 1: Check first character
    if text[0] == "1": return 1
    if text[0] == "0": return 0

    # Step 2: Check for Bengali yes/no words
    bengali_yes = ["হ্যাঁ", "হাঁ", "জি", "ঠিক", "অবশ্যই"]
    bengali_no  = ["না", "না।", "নহয়", "নয়"]
    for w in bengali_yes:
        if text.startswith(w): return 1
    for w in bengali_no:
        if text.startswith(w): return 0

    # Step 3: Check for English yes/no
    text_lower = text.lower()
    english_yes = ["yes", "true"]
    english_no  = ["no", "false"]
    for w in english_yes:
        if text_lower.startswith(w): return 1
    for w in english_no:
        if text_lower.startswith(w): return 0

    # Step 4: Scan for Bengali digits (first occurrence)
    bengali_map = {"\u09e7": 1, "\u09e6": 0}  # ১, ০
    for ch in text:
        if ch in bengali_map: return bengali_map[ch]

    # Step 5: Look for the LAST ASCII digit 0/1 in the text
    # (LLMs often reason: "The article uses sensational language... therefore 1")
    last_digit = None
    for ch in reversed(text):
        if ch in "01":
            last_digit = int(ch)
            break
    if last_digit is not None:
        return last_digit

    # Step 6: Scan for any ASCII digit 0 or 1 (first occurrence, safety net)
    for ch in text:
        if ch == "1": return 1
        if ch == "0": return 0

    return -1


def compute_metrics(y_true, y_pred, model_name, n_unparseable=0):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    parseable_pct = 100.0 * (len(y_true) - n_unparseable) / max(len(y_true), 1)
    print(f"\n{'='*55}", flush=True)
    print(f"  {model_name}", flush=True)
    print(f"{'='*55}", flush=True)
    print(f"  Acc={acc:.4f}  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}", flush=True)
    print(f"  Kappa={kappa:.4f}  MCC={mcc:.4f}", flush=True)
    print(f"  TP={tp} FP={fp} FN={fn} TN={tn}", flush=True)
    print(f"  Parseable: {len(y_true)-n_unparseable}/{len(y_true)} ({parseable_pct:.1f}%)", flush=True)
    print(f"{'='*55}", flush=True)
    return {"Model": model_name, "Accuracy": round(acc,4), "Precision": round(prec,4),
            "Recall": round(rec,4), "F1": round(f1,4), "Kappa": round(kappa,4),
            "MCC": round(mcc,4), "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
            "Unparseable": int(n_unparseable), "Parseable_Pct": round(parseable_pct,1)}

def cleanup_gpu():
    """Aggressive GPU cleanup — prevents CUDA OOM between seeds."""
    import gc
    import torch

    # Multiple gc.collect passes (Python uses reference counting + GC)
    for _ in range(3):
        gc.collect()

    if torch.cuda.is_available():
        # Synchronize all async operations before clearing
        torch.cuda.synchronize()

        # Clear cache multiple times (some allocations are deferred)
        for _ in range(3):
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        # Reset peak memory stats for accurate monitoring
        torch.cuda.reset_peak_memory_stats()

        # Print memory status for debugging
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        print(f"[{_now()}] GPU cleanup done | allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB", flush=True)

        # Critical: if allocated memory is still high, the next model load will OOM
        if allocated > 1.0:
            print(f"[{_now()}] WARNING: GPU still has {allocated:.2f} GB allocated — next seed may OOM", flush=True)
            print(f"[{_now()}] Attempting additional cleanup...", flush=True)
            # Force another round
            for _ in range(5):
                gc.collect()
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            allocated = torch.cuda.memory_allocated(0) / 1e9
            print(f"[{_now()}] After additional cleanup: {allocated:.2f} GB allocated", flush=True)


def cleanup_model_files(model_path, model_name):
    """Delete model files to free disk space. Only deletes HuggingFace
    cache, not Kaggle Input files (those are read-only)."""
    if model_path.startswith("/kaggle/input/"):
        print(f"[{_now()}] Kaggle Input model — not deleting (read-only)", flush=True)
        return
    # HuggingFace cache cleanup
    cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
    if os.path.exists(cache_dir):
        cache_size = sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fns in os.walk(cache_dir) for f in fns) / 1e9
        shutil.rmtree(cache_dir, ignore_errors=True)
        print(f"[{_now()}] HF cache cleared: {cache_size:.2f} GB freed", flush=True)
    # Local lora output
    lora_dir = f"./lora_{model_name.replace('/', '_')}"
    if os.path.exists(lora_dir):
        shutil.rmtree(lora_dir, ignore_errors=True)

def plot_cm(y_true, y_pred, model_name, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not YJ","Yellow J."], yticklabels=["Not YJ","Yellow J."])
    plt.title(model_name, fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

print(f"[{_now()}] Utilities ready.", flush=True)


# ============================================================
# finetune_and_evaluate — parameterised on `seed`
# ============================================================

def finetune_and_evaluate(model_id, model_name, seed):
    """Full pipeline for one seed: Load 4-bit -> Format -> LoRA -> SFT -> Eval -> Cleanup.

    Parameters
    ----------
    model_id : str
        HuggingFace model ID (used for HF-Hub fallback; Kaggle path is resolved
        from the global MODEL_CONFIG).
    model_name : str
        Display name (used in logs, metrics, and output filenames).
    seed : int
        Random seed for set_seed(), train_test_split(), and TrainingArguments.
    """
    # --- 0. Seed EVERYTHING first ---------------------------------------
    set_seed(seed)
    tag = f"{model_name}_seed{seed}"
    # Pre-load memory check (Task 12 fix: detect residual GPU pressure from
    # previous seed before attempting model load).
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e9
        free, total = torch.cuda.mem_get_info(0)
        free_gb = free / 1e9
        total_gb = total / 1e9
        print(f"[{_now()}] [{tag}] Pre-load GPU memory: {allocated:.2f} GB allocated, {free_gb:.2f} / {total_gb:.2f} GB free", flush=True)
        if free_gb < 4.0:
            print(f"[{_now()}] [{tag}] WARNING: Low free GPU memory ({free_gb:.2f} GB) — may OOM during model load", flush=True)
            print(f"[{_now()}] [{tag}] Attempting pre-emptive cleanup...", flush=True)
            import gc as _gc_pre
            for _ in range(3):
                _gc_pre.collect()
            torch.cuda.synchronize()
            for _ in range(3):
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            free, total = torch.cuda.mem_get_info(0)
            free_gb = free / 1e9
            print(f"[{_now()}] [{tag}] After pre-emptive cleanup: {free_gb:.2f} GB free", flush=True)

    print(f"\n[{_now()}] {'='*60}", flush=True)
    print(f"[{_now()}] {tag}", flush=True)
    print(f"[{_now()}] {'='*60}", flush=True)
    t0 = time.time()

    # --- 0a. Resolve model path (Kaggle preferred, HF fallback) ---------
    model_path, model_source = find_model_path(MODEL_CONFIG)
    print(f"[{_now()}] [{tag}] Source: {model_path} [{model_source}]", flush=True)

    # --- 0b. Per-model memory tuning (7B override, same as NB5) ---------
    global MAX_SEQ_LEN, BATCH_SIZE
    if "9B" in model_name or "8B" in model_name:
        MAX_SEQ_LEN = 256
        BATCH_SIZE = 1
        print(f"[{_now()}] [{tag}] Large model — MAX_SEQ_LEN={MAX_SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}", flush=True)
    elif "7B" in model_name:
        MAX_SEQ_LEN = 320
        BATCH_SIZE = 1
        print(f"[{_now()}] [{tag}] 7B model — MAX_SEQ_LEN={MAX_SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}", flush=True)
    else:
        MAX_SEQ_LEN = 384
        BATCH_SIZE = 2
    sys.stdout.flush()

    # --- 0c. Per-seed stratified train/test split -----------------------
    # This is the primary source of run-to-run variance: different seeds
    # produce different 612/154 partitions.
    train_df, test_df = train_test_split(
        df[[TEXT_COL, LABEL_COL]], test_size=1-TRAIN_FRAC,
        stratify=df[LABEL_COL], random_state=seed
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    print(f"[{_now()}] [{tag}] Split: train={len(train_df)} test={len(test_df)} (seed={seed})", flush=True)
    print(f"[{_now()}] [{tag}] Train labels: {train_df[LABEL_COL].value_counts().sort_index().to_dict()}", flush=True)
    print(f"[{_now()}] [{tag}] Test  labels: {test_df[LABEL_COL].value_counts().sort_index().to_dict()}", flush=True)

    # --- 1. Tokenizer ---------------------------------------------------
    print(f"[{_now()}] [{tag}] 1/6 Loading tokenizer...", flush=True)
    tok_kwargs = {"trust_remote_code": True}
    if os.environ.get("HF_TOKEN"):
        tok_kwargs["token"] = os.environ["HF_TOKEN"]
    tokenizer = AutoTokenizer.from_pretrained(model_path, **tok_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # FIX 1: Chat template fallback — some tokenizers don't ship one.
    if not hasattr(tokenizer, 'chat_template') or tokenizer.chat_template is None:
        print(f"[{_now()}] [{tag}] No chat_template found — using default.", flush=True)
        tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'system' %}{{ message['content'] }}\n{% elif message['role'] == 'user' %}User: {{ message['content'] }}\n{% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }}\n{% endif %}{% endfor %}Assistant: "

    # --- 2. Model (4-bit, fp16) -----------------------------------------
    print(f"[{_now()}] [{tag}] 2/6 Loading model (4-bit NF4, fp16)...", flush=True)
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs = {
        "quantization_config": bnb_cfg,
        "device_map": {"": 0},
        "trust_remote_code": True,
        "torch_dtype": torch.float16,
    }
    if os.environ.get("HF_TOKEN"):
        model_kwargs["token"] = os.environ["HF_TOKEN"]
    model = AutoModelForCausalLM.from_pretrained(model_path, **model_kwargs)
    # FIX 2: Force float16 — some models load as bfloat16, causing AMP errors on T4
    model.config.torch_dtype = torch.float16
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model.enable_input_require_grads()
    print(f"[{_now()}] [{tag}] VRAM: {torch.cuda.memory_allocated(0)/1e9:.2f} GB", flush=True)
    sys.stdout.flush()

    # --- 3. Format data -------------------------------------------------
    print(f"[{_now()}] [{tag}] 3/6 Formatting data...", flush=True)
    def _fmt(row):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(row[TEXT_COL])[:2000]},
            {"role": "assistant", "content": str(int(row[LABEL_COL]))},
        ]
        return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}
    tr_ds = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]])
    tr_ds = tr_ds.map(_fmt, remove_columns=tr_ds.column_names)

    te_prompts, te_labels = [], []
    for _, row in test_df.iterrows():
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(row[TEXT_COL])[:2000]},
        ]
        te_prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        te_labels.append(int(row[LABEL_COL]))

    # --- 4. LoRA --------------------------------------------------------
    print(f"[{_now()}] [{tag}] 4/6 Setting up LoRA...", flush=True)
    preferred = ["q_proj", "k_proj", "v_proj", "o_proj"]
    named = set(n for n, _ in model.named_modules())
    targets = [m for m in preferred if any(m in n for n in named)]
    if not targets: targets = ["q_proj", "v_proj"]
    print(f"[{_now()}] [{tag}] LoRA targets: {targets}", flush=True)

    lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=targets,
                          lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
    args = TrainingArguments(
        output_dir=f"./lora_{model_name.replace('/', '_')}_seed{seed}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        fp16=False,
        bf16=False,
        logging_steps=25,
        disable_tqdm=False,
        report_to="none",
        save_strategy="no",
        gradient_checkpointing=False,
        optim="paged_adamw_8bit",
        seed=seed,
        data_seed=seed,
        dataloader_pin_memory=False,
        max_grad_norm=1.0,
    )

    # Custom callback for real-time progress logging in Kaggle commit mode
    from transformers import TrainerCallback
    class FlushProgressCallback(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs is not None:
                epoch = logs.get('epoch', 0)
                loss = logs.get('loss', 0)
                lr = logs.get('learning_rate', 0)
                step = state.global_step
                total = state.max_steps
                pct = 100 * step / total if total > 0 else 0
                print(f"[{_now()}] [{tag}] step {step}/{total} ({pct:.1f}%) | "
                      f"epoch={epoch:.2f} | loss={loss:.4f} | lr={lr:.2e}", flush=True)

    # Version-safe SFTTrainer
    sig = set(inspect.signature(SFTTrainer.__init__).parameters)
    tr_kwargs = {"model": model, "train_dataset": tr_ds, "args": args, "peft_config": lora_cfg}
    if "processing_class" in sig: tr_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig: tr_kwargs["tokenizer"] = tokenizer
    if "max_seq_length" in sig: tr_kwargs["max_seq_length"] = MAX_SEQ_LEN
    elif "max_seq_len" in sig: tr_kwargs["max_seq_len"] = MAX_SEQ_LEN
    if "dataset_text_field" in sig: tr_kwargs["dataset_text_field"] = "text"
    trainer = SFTTrainer(**tr_kwargs)
    trainer.add_callback(FlushProgressCallback())

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"[{_now()}] [{tag}] Trainable: {trainable:,}/{total:,} ({100*trainable/total:.2f}%)", flush=True)
    sys.stdout.flush()

    # --- 5. Train -------------------------------------------------------
    print(f"[{_now()}] [{tag}] 5/6 Training {NUM_EPOCHS} epoch(s)...", flush=True)
    t_train = time.time()
    trainer.train()
    train_min = (time.time() - t_train) / 60
    print(f"[{_now()}] [{tag}] Training: {train_min:.1f} min", flush=True)
    sys.stdout.flush()

    # --- 6. Evaluate ----------------------------------------------------
    print(f"[{_now()}] [{tag}] 6/6 Evaluating {len(te_prompts)} samples...", flush=True)
    t_eval = time.time()
    preds, unparseable = [], 0
    model.eval()
    for i, prompt in enumerate(te_prompts):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=3, do_sample=False,
                                 temperature=0.0, pad_token_id=tokenizer.pad_token_id)
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        p = parse_prediction(gen)
        preds.append(p if p != -1 else 0)
        if p == -1:
            unparseable += 1
            # Log first 10 unparseable outputs for debugging
            if unparseable <= 10:
                print(f"[{_now()}] [{tag}] Unparseable output #{unparseable}: repr={repr(gen[:200])}", flush=True)
        if (i+1) % 100 == 0:
            el = (time.time()-t_eval)/60
            print(f"[{_now()}] [{tag}] {i+1}/{len(te_prompts)} ({el:.1f}m, unparseable={unparseable})", flush=True)
    eval_min = (time.time()-t_eval)/60

    metrics = compute_metrics(te_labels, preds, model_name, unparseable)
    metrics["Train_Min"] = round(train_min, 1)
    metrics["Eval_Min"] = round(eval_min, 1)
    metrics["Source"] = model_source
    metrics["Total_Min"] = round((time.time()-t0)/60, 1)
    metrics["MAX_SEQ_LEN"] = MAX_SEQ_LEN
    metrics["BATCH_SIZE"] = BATCH_SIZE

    safe = model_name.replace("/", "_").replace(".", "_")
    plot_cm(te_labels, preds, f"{model_name} (seed={seed})",
            f"{OUTPUT_DIR}/cm_{safe}_seed{seed}.png")
    print(f"[{_now()}] [{tag}] TOTAL: {metrics['Total_Min']:.1f} min", flush=True)

    # === AGGRESSIVE CLEANUP (prevents CUDA OOM between seeds) ===
    print(f"[{_now()}] [{tag}] Starting aggressive cleanup...", flush=True)

    # Delete in reverse order of creation (dependencies first).
    # Note: `del locals()[x]` is a no-op in CPython — locals() returns a
    # snapshot. We keep the loop for visibility AND follow it with direct
    # `del` statements that actually release GPU tensors.
    objects_to_delete = ['trainer', 'model', 'tokenizer', 'tr_ds', 'te_prompts', 'te_labels']
    for obj_name in objects_to_delete:
        if obj_name in locals():
            try:
                del locals()[obj_name]
            except Exception:
                pass

    # Working deletions (the loop above is a no-op due to CPython locals()
    # semantics — fast locals cannot be deleted via the locals() dict).
    for obj_name in objects_to_delete:
        try:
            if obj_name == 'trainer':    del trainer
            elif obj_name == 'model':    del model
            elif obj_name == 'tokenizer': del tokenizer
            elif obj_name == 'tr_ds':    del tr_ds
            elif obj_name == 'te_prompts': del te_prompts
            elif obj_name == 'te_labels': del te_labels
        except NameError:
            pass
        except Exception:
            pass

    # Also delete any LoRA adapter files on disk (spec pattern + the
    # seed-suffixed dir that TrainingArguments actually creates).
    import shutil
    lora_dir = f"./lora_{model_name.replace('/', '_')}"
    if os.path.exists(lora_dir):
        shutil.rmtree(lora_dir, ignore_errors=True)
        print(f"[{_now()}] [{tag}] Deleted LoRA directory: {lora_dir}", flush=True)
    lora_dir_seed = f"./lora_{model_name.replace('/', '_')}_seed{seed}"
    if os.path.exists(lora_dir_seed):
        shutil.rmtree(lora_dir_seed, ignore_errors=True)
        print(f"[{_now()}] [{tag}] Deleted LoRA directory: {lora_dir_seed}", flush=True)

    # Force garbage collection
    import gc
    for _ in range(3):
        gc.collect()

    # Clear CUDA cache
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        for _ in range(3):
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        print(f"[{_now()}] [{tag}] Cleanup complete | allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB", flush=True)

    # CRITICAL: Sleep briefly to let the OS release memory
    time.sleep(5)  # time is imported at module level

    sys.stdout.flush()
    return metrics

print(f"[{_now()}] finetune_and_evaluate(model_id, model_name, seed) ready.", flush=True)


[17:42:00] Utilities ready.
[17:42:00] finetune_and_evaluate(model_id, model_name, seed) ready.


### 6. Multi-Seed Runner (Batch 1)

Iterates over `SEEDS = [42, 123, 2024]` and calls `finetune_and_evaluate()` for each seed. After each seed completes (or fails), the intermediate results are saved to `multi_seed_qwen7b_batch1_results.csv` — this is crash-safe: if the kernel dies mid-way through seed 2, the results for seed 1 are already persisted. A single seed failure does not terminate the experiment; the failure is recorded with an `error` field and the loop continues.


In [6]:
# ============================================================
# MULTI-SEED RUNNER — Batch 1/2 (seeds 42, 123, 2024)
# ============================================================
import sys
import traceback

all_results = []
overall_t0 = time.time()

for i, seed in enumerate(SEEDS):
    print(f"\n[{_now()}] === Seed {i+1}/{len(SEEDS)} (Batch 1): {seed} ===", flush=True)
    try:
        metrics = finetune_and_evaluate(
            model_id=MODEL_CONFIG["hf_id"],
            model_name=MODEL_CONFIG["name"],
            seed=seed,
        )
        metrics["seed"] = seed
        all_results.append(metrics)
        print(f"\n[{_now()}] >>> DONE seed={seed}: "
              f"Acc={metrics['Accuracy']}, F1={metrics['F1']}", flush=True)
    except Exception as e:
        print(f"[{_now()}] Seed {seed} FAILED: {e}", flush=True)
        traceback.print_exc()
        sys.stdout.flush()
        all_results.append({"seed": seed, "error": str(e)})
    # Save intermediate results after each seed (crash-safe)
    pd.DataFrame(all_results).to_csv(RESULTS_FILE, index=False)
    print(f"[{_now()}] Saved intermediate results to {RESULTS_FILE}", flush=True)

total_min = (time.time() - overall_t0) / 60
print(f"\n[{_now()}] BATCH 1 DONE in {total_min:.1f} min ({total_min/60:.1f} hours)", flush=True)
print(f"[{_now()}] Successful: {sum(1 for r in all_results if 'error' not in r)}/{len(SEEDS)}", flush=True)
print(f"[{_now()}] Remaining seeds (handled by NB9b): {REMAINING_SEEDS}", flush=True)



[17:42:00] === Seed 1/3 (Batch 1): 42 ===
[17:42:00] [Qwen2.5-7B-Instruct_seed42] Pre-load GPU memory: 0.00 GB allocated, 15.53 / 15.64 GB free

[17:42:00] ============================================================
[17:42:00] Qwen2.5-7B-Instruct_seed42
[17:42:00] ============================================================
[17:42:00] [Qwen2.5-7B-Instruct_seed42] Source: /kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1 [kaggle]
[17:42:00] [Qwen2.5-7B-Instruct_seed42] 7B model — MAX_SEQ_LEN=320, BATCH_SIZE=1
[17:42:00] [Qwen2.5-7B-Instruct_seed42] Split: train=612 test=154 (seed=42)
[17:42:00] [Qwen2.5-7B-Instruct_seed42] Train labels: {0: 306, 1: 306}
[17:42:00] [Qwen2.5-7B-Instruct_seed42] Test  labels: {0: 77, 1: 77}
[17:42:00] [Qwen2.5-7B-Instruct_seed42] 1/6 Loading tokenizer...
[17:42:01] [Qwen2.5-7B-Instruct_seed42] 2/6 Loading model (4-bit NF4, fp16)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[17:42:28] [Qwen2.5-7B-Instruct_seed42] VRAM: 7.73 GB
[17:42:28] [Qwen2.5-7B-Instruct_seed42] 3/6 Formatting data...


Map:   0%|          | 0/612 [00:00<?, ? examples/s]

[17:42:30] [Qwen2.5-7B-Instruct_seed42] 4/6 Setting up LoRA...
[17:42:30] [Qwen2.5-7B-Instruct_seed42] LoRA targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj']


...

Building labels for train dataset:   0%|          | 0/612 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/612 [00:00<?, ? examples/s]

[20:16:47] [Qwen2.5-7B-Instruct_seed2024] Trainable: 10,092,544/4,363,064,832 (0.23%)
[20:16:47] [Qwen2.5-7B-Instruct_seed2024] 5/6 Training 3 epoch(s)...


Step,Training Loss
25,1.156512
50,0.953421
75,0.919743
100,0.895535
125,0.899459
150,0.899373
175,0.871569
200,0.852068
225,0.854001
250,0.837853


[20:25:11] [Qwen2.5-7B-Instruct_seed2024] step 25/459 (5.4%) | epoch=0.16 | loss=1.1565 | lr=2.00e-04
[20:33:26] [Qwen2.5-7B-Instruct_seed2024] step 50/459 (10.9%) | epoch=0.33 | loss=0.9534 | lr=1.97e-04
[20:41:41] [Qwen2.5-7B-Instruct_seed2024] step 75/459 (16.3%) | epoch=0.49 | loss=0.9197 | lr=1.91e-04
[20:49:46] [Qwen2.5-7B-Instruct_seed2024] step 100/459 (21.8%) | epoch=0.65 | loss=0.8955 | lr=1.83e-04
[20:58:00] [Qwen2.5-7B-Instruct_seed2024] step 125/459 (27.2%) | epoch=0.82 | loss=0.8995 | lr=1.71e-04
[21:06:20] [Qwen2.5-7B-Instruct_seed2024] step 150/459 (32.7%) | epoch=0.98 | loss=0.8994 | lr=1.58e-04
[21:14:37] [Qwen2.5-7B-Instruct_seed2024] step 175/459 (38.1%) | epoch=1.14 | loss=0.8716 | lr=1.43e-04
[21:22:55] [Qwen2.5-7B-Instruct_seed2024] step 200/459 (43.6%) | epoch=1.31 | loss=0.8521 | lr=1.26e-04
[21:31:15] [Qwen2.5-7B-Instruct_seed2024] step 225/459 (49.0%) | epoch=1.47 | loss=0.8540 | lr=1.09e-04
[21:39:20] [Qwen2.5-7B-Instruct_seed2024] step 250/459 (54.5%) | epo

### 7. Aggregate Statistics (Batch 1)

Loads the saved CSV, filters out any failed runs (those with an `error` field), and computes mean / std / min / max across the 3 seeds in this batch for each metric. The full 5-seed aggregation is performed by NB9c after both batches complete.


In [7]:
# ============================================================
# AGGREGATE STATISTICS — Batch 1 (3 seeds)
# ============================================================

# Reload from CSV (so this cell works even if the runner cell is re-run
# in a fresh kernel after a completed commit).
results_df = pd.read_csv(RESULTS_FILE) if os.path.exists(RESULTS_FILE) else pd.DataFrame(all_results)
print(f"[{_now()}] Loaded {len(results_df)} rows from {RESULTS_FILE}", flush=True)

print(f"\n[{_now()}] Raw per-seed results (Batch 1):", flush=True)
print(results_df.to_string(index=False), flush=True)

# Filter out failed runs (those with an 'error' column populated).
# Use a robust check: a row is "successful" if it has no 'error' column
# OR its 'error' value is NaN.
if 'error' in results_df.columns:
    successful = results_df[results_df['error'].isna()].copy()
else:
    successful = results_df.copy()

# Coerce metric columns to numeric (CSV round-trip can turn them into strings)
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'Kappa', 'MCC']
for c in metric_cols:
    if c in successful.columns:
        successful[c] = pd.to_numeric(successful[c], errors='coerce')

metric_cols_present = [c for c in metric_cols if c in successful.columns and successful[c].notna().any()]

print("\n" + "=" * 70, flush=True)
print(f"  Multi-Seed Results (Qwen2.5-7B-Instruct, Batch 1, {len(SEEDS)} seeds)", flush=True)
print("=" * 70, flush=True)
if len(successful) == 0:
    print("  ⚠️  No successful runs to aggregate!", flush=True)
    print("  All seeds failed. Check the error column in the results CSV.", flush=True)
    agg = None
else:
    agg = successful[metric_cols_present].agg(['mean', 'std', 'min', 'max'])
    print(agg.round(4).to_string(), flush=True)
print("=" * 70, flush=True)

if 'F1' in successful.columns and len(successful) > 0:
    print(f"\nF1:       {successful['F1'].mean():.4f} ± {successful['F1'].std():.4f}", flush=True)
if 'Accuracy' in successful.columns and len(successful) > 0:
    print(f"Accuracy: {successful['Accuracy'].mean():.4f} ± {successful['Accuracy'].std():.4f}", flush=True)
print(f"\nN successful runs (Batch 1): {len(successful)} / {len(SEEDS)}", flush=True)


[22:50:28] Loaded 3 rows from /kaggle/working/multi_seed_qwen7b_batch1_results.csv

[22:50:28] Raw per-seed results (Batch 1):
              Model  Accuracy  Precision  Recall     F1  Kappa    MCC  TP  FP   FN   TN  Unparseable  Parseable_Pct  Train_Min  Eval_Min Source  Total_Min  MAX_SEQ_LEN  BATCH_SIZE  seed                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     error
Qwen2.5-7B-Instruct    0.5130     1.0000   0.026 0.0506  0.026 0.1147 2.0 0.0 75.0 77.0        151.0            1.9      151.6       2.0 kaggle      

### 8. Visualisation (Batch 1)

Bar plot of F1 per seed in this batch, with an error bar showing ±1 std across the 3 seeds and a horizontal line at the batch mean. The plot is saved to `/kaggle/working/multi_seed_f1_batch1.png`. The full 5-seed plot (across both batches) is produced by NB9c.


In [8]:
# ============================================================
# VISUALISATION — F1 per seed (Batch 1)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

if len(successful) > 0 and 'F1' in successful.columns:
    seeds = successful['seed'].astype(str).tolist()
    f1_vals = successful['F1'].astype(float).values
    f1_mean = float(np.mean(f1_vals))
    f1_std = float(np.std(f1_vals, ddof=1)) if len(f1_vals) > 1 else 0.0

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(seeds, f1_vals,
                  yerr=[f1_std] * len(seeds),
                  capsize=8, color='steelblue', edgecolor='black', alpha=0.85,
                  error_kw={'elinewidth': 1.5, 'ecolor': 'dimgray'})
    ax.axhline(f1_mean, color='red', ls='--', lw=1.5,
               label=f"Batch-1 Mean F1 = {f1_mean:.4f} ± {f1_std:.4f}")
    ax.set_xlabel("Seed", fontsize=12)
    ax.set_ylabel("F1 Score", fontsize=12)
    ax.set_title("Qwen2.5-7B-Instruct — F1 per Seed (Batch 1, 3 seeds)\n"
                 "(Full 5-seed plot produced by NB9c)",
                 fontsize=11)
    y_max = max(0.15, float(np.max(f1_vals)) * 1.4 + f1_std)
    ax.set_ylim(0, y_max)
    ax.legend(fontsize=10, loc='upper right')
    for bar, v in zip(bars, f1_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.003,
                f"{v:.4f}", ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    out_png = f"{OUTPUT_DIR}/multi_seed_f1_batch1.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[{_now()}] Saved: {out_png}", flush=True)
else:
    print(f"[{_now()}] No successful runs — skipping plot.", flush=True)


[22:50:28] Saved: /kaggle/working/multi_seed_f1_batch1.png


### 9. Comparison with Single-Seed NB5 Result (Batch 1)

The original NB5 reported a single-seed F1 of **0.075** for Qwen2.5-7B-Instruct (`seed=42`). This cell compares that single-seed number to the 3-seed mean±std from NB9a Batch 1. The full 5-seed comparison (across both batches) is performed by NB9c.


In [9]:
# ============================================================
# COMPARE — NB5 single-seed vs NB9a Batch 1 (3-seed mean±std)
# ============================================================

NB5_F1 = 0.075  # single-seed F1 from NB5 (seed=42, Qwen2.5-7B-Instruct)

if len(successful) > 0 and 'F1' in successful.columns:
    multi_f1_mean = float(successful['F1'].mean())
    multi_f1_std = float(successful['F1'].std()) if len(successful) > 1 else 0.0
    multi_f1_min = float(successful['F1'].min())
    multi_f1_max = float(successful['F1'].max())
else:
    multi_f1_mean = float('nan')
    multi_f1_std = float('nan')
    multi_f1_min = float('nan')
    multi_f1_max = float('nan')

print("=" * 70, flush=True)
print("  NB5 (single seed=42) vs NB9a Batch 1 (3-seed mean±std)", flush=True)
print("=" * 70, flush=True)
print(f"  NB5         F1 (single seed=42): {NB5_F1:.4f}", flush=True)
print(f"  NB9a Batch1 F1 (mean ± std):    {multi_f1_mean:.4f} ± {multi_f1_std:.4f}", flush=True)
print(f"  NB9a Batch1 F1 range:           [{multi_f1_min:.4f}, {multi_f1_max:.4f}]", flush=True)
print("=" * 70, flush=True)

within_1std = abs(NB5_F1 - multi_f1_mean) <= multi_f1_std if multi_f1_std > 0 else False
within_2std = abs(NB5_F1 - multi_f1_mean) <= 2 * multi_f1_std if multi_f1_std > 0 else False
print(f"\n  NB5 F1 within 1 std of NB9a Batch-1 mean: {within_1std}", flush=True)
print(f"  NB5 F1 within 2 std of NB9a Batch-1 mean: {within_2std}", flush=True)

print(f"\n  NOTE: Full 5-seed comparison vs NB5 is computed by NB9c after both", flush=True)
print(f"        batches complete. The values above are Batch-1 only.", flush=True)

# Per-seed breakdown vs NB5
if len(successful) > 0 and 'F1' in successful.columns:
    print(f"\n  Per-seed F1 (Batch 1) vs NB5 single-seed F1={NB5_F1:.4f}:", flush=True)
    for _, row in successful.iterrows():
        diff = float(row['F1']) - NB5_F1
        print(f"    seed={int(row['seed']):<5}  F1={float(row['F1']):.4f}  (Δ={diff:+.4f})", flush=True)


  NB5 (single seed=42) vs NB9a Batch 1 (3-seed mean±std)
  NB5         F1 (single seed=42): 0.0750
  NB9a Batch1 F1 (mean ± std):    0.0503 ± 0.0004
  NB9a Batch1 F1 range:           [0.0500, 0.0506]

  NB5 F1 within 1 std of NB9a Batch-1 mean: False
  NB5 F1 within 2 std of NB9a Batch-1 mean: False

  NOTE: Full 5-seed comparison vs NB5 is computed by NB9c after both
        batches complete. The values above are Batch-1 only.

  Per-seed F1 (Batch 1) vs NB5 single-seed F1=0.0750:
    seed=42     F1=0.0506  (Δ=-0.0244)
    seed=2024   F1=0.0500  (Δ=-0.0250)


### 10. Save Batch 1 Summary

Writes a JSON summary to `/kaggle/working/multi_seed_qwen7b_batch1_summary.json` containing the model name, seed list, per-seed results, aggregate mean/std for each metric, and batch provenance fields (`batch=1`, `n_seeds_in_batch=3`, `total_seeds_planned=5`, `remaining_seeds=[7, 99]`). This file is consumed by NB9c to produce the final 5-seed aggregated summary.


In [10]:
# ============================================================
# SAVE BATCH 1 SUMMARY JSON
# ============================================================

def _safe_mean(col):
    if len(successful) > 0 and col in successful.columns:
        return float(pd.to_numeric(successful[col], errors='coerce').mean())
    return None

def _safe_std(col):
    if len(successful) > 1 and col in successful.columns:
        return float(pd.to_numeric(successful[col], errors='coerce').std())
    return None

summary = {
    "model": "Qwen2.5-7B-Instruct",
    "batch": 1,
    "n_seeds_in_batch": len(SEEDS),
    "seeds": SEEDS,
    "total_seeds_planned": TOTAL_SEEDS_PLANNED,
    "remaining_seeds": REMAINING_SEEDS,
    "n_successful": int(len(successful)) if 'successful' in globals() else 0,
    "f1_mean": _safe_mean('F1'),
    "f1_std": _safe_std('F1'),
    "accuracy_mean": _safe_mean('Accuracy'),
    "accuracy_std": _safe_std('Accuracy'),
    "precision_mean": _safe_mean('Precision'),
    "recall_mean": _safe_mean('Recall'),
    "kappa_mean": _safe_mean('Kappa'),
    "mcc_mean": _safe_mean('MCC'),
    "per_seed_results": all_results,
    "results_csv": RESULTS_FILE,
    "note": ("Batch 1 of 2 of multi-seed evaluation to estimate variance of "
             "QLoRA fine-tuning on Qwen2.5-7B-Instruct for Bengali yellow "
             "journalism detection. Batch 1 covers seeds [42, 123, 2024]; "
             "Batch 2 (NB9b) covers seeds [7, 99]. Run NB9c to aggregate "
             "both batches into the final 5-seed mean±std. Split because "
             "the full 5-seed run (~13h) exceeds Kaggle's 12-hour commit "
             "limit. Addresses reviewer concern about single-seed evaluation in NB5."),
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)
print(f"[{_now()}] Saved summary: {SUMMARY_FILE}", flush=True)

# Also rewrite the per-seed CSV (final version)
pd.DataFrame(all_results).to_csv(RESULTS_FILE, index=False)
print(f"[{_now()}] Saved per-seed CSV: {RESULTS_FILE}", flush=True)

print(f"\n[{_now()}] Output files in {OUTPUT_DIR}:", flush=True)
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name:<50} {f.stat().st_size/1024:>8.1f} KB", flush=True)


[22:50:28] Saved summary: /kaggle/working/multi_seed_qwen7b_batch1_summary.json
[22:50:28] Saved per-seed CSV: /kaggle/working/multi_seed_qwen7b_batch1_results.csv

[22:50:28] Output files in /kaggle/working:
  __notebook__.ipynb                                    114.4 KB
  cm_Qwen2_5-7B-Instruct_seed2024.png                    25.6 KB
  cm_Qwen2_5-7B-Instruct_seed42.png                      25.5 KB
  multi_seed_f1_batch1.png                               49.3 KB
  multi_seed_qwen7b_batch1_results.csv                    0.9 KB
  multi_seed_qwen7b_batch1_summary.json                   2.5 KB


### 11. Next Steps

After this notebook completes:

1. Download `multi_seed_qwen7b_batch1_results.csv` from the output panel.
2. Upload it as a Kaggle dataset (e.g., named `swarabyanjan-qwen7b-batch1`).
3. Run `NB9b_Qwen-7B_seeds_7_99.ipynb` (Batch 2).
4. Run `NB9c_Qwen-7B_aggregate.ipynb` to combine batches and produce the final summary + plots.
